<div dir="rtl">

# الدرس 30: اختيار العتبة، ومنحنى الأداء

دفتر الكود لهذا الدرس. شغّل الخلايا بالترتيب من الأعلى.

</div>

<div dir="rtl">

## بالكود

</div>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

X, y_raw = load_breast_cancer(return_X_y=True)
y = 1 - y_raw                       # 1 = خبيث
X_tmp, X_te, y_tmp, y_te = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
X_tr, X_val, y_tr, y_val = train_test_split(X_tmp, y_tmp, test_size=0.33, random_state=0, stratify=y_tmp)

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_tr, y_tr)
p_val = model.predict_proba(X_val)[:, 1]          # احتمال الخبيث
p_te = model.predict_proba(X_te)[:, 1]

<div dir="rtl">

### العتبة تغيّر كل شيء

</div>

In [ ]:
from sklearn.metrics import precision_score, recall_score

for t in [0.1, 0.3, 0.5, 0.7, 0.9]:
    pred = (p_val >= t).astype(int)
    print(f"threshold {t:.1f}: precision = {precision_score(y_val, pred):.3f}  recall = {recall_score(y_val, pred):.3f}")

<div dir="rtl">

### المنحنيان

</div>

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve, average_precision_score

fpr, tpr, _ = roc_curve(y_val, p_val)
prec, rec, _ = precision_recall_curve(y_val, p_val)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.8))
a1.plot(fpr, tpr, lw=2.5, label=f"AUC = {roc_auc_score(y_val, p_val):.3f}")
a1.plot([0, 1], [0, 1], "--", color="gray")
a1.set_xlabel("false positive rate"); a1.set_ylabel("true positive rate (recall)"); a1.legend(); a1.grid(True)
a2.plot(rec, prec, lw=2.5, color="C1", label=f"AP = {average_precision_score(y_val, p_val):.3f}")
a2.axhline(y_val.mean(), ls="--", color="gray")
a2.set_xlabel("recall"); a2.set_ylabel("precision"); a2.legend(); a2.grid(True)
plt.tight_layout()
plt.show()

<div dir="rtl">

### نختار العتبة حسب الكلفة

</div>

In [ ]:
cost_fn, cost_fp = 20, 1
best_t, best_cost = None, None
for t in np.linspace(0.01, 0.99, 99):
    pred = p_val >= t
    fn = np.sum(~pred & (y_val == 1))
    fp = np.sum(pred & (y_val == 0))
    cost = cost_fn * fn + cost_fp * fp
    if best_cost is None or cost < best_cost:
        best_t, best_cost = t, cost
print(f"best threshold on validation: {best_t:.2f} (cost {best_cost})")

for t in [0.5, best_t]:
    pred = (p_te >= t).astype(int)
    fn = int(np.sum((pred == 0) & (y_te == 1))); fp = int(np.sum((pred == 1) & (y_te == 0)))
    print(f"test, threshold {t:.2f}: missed cancers = {fn}, extra checks = {fp}, cost = {cost_fn * fn + cost_fp * fp}")

<div dir="rtl">

## تمرين: ست درجات

| المثال | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|
| الدرجة | 0.9 | 0.8 | 0.6 | 0.4 | 0.35 | 0.2 |
| الحقيقة | 1 | 0 | 1 | 1 | 0 | 0 |

1. احسب معدل الإيجابي الصحيح والكاذب عند العتبات: 0.85، و 0.5، و 0.3.
2. احسب المساحة تحت المنحنى بعدّ الأزواج.
3. عند أي عتبة من الثلاث يكون الاستدعاء كاملاً؟ وما الدقة الإيجابية عندها؟
4. تحقق من المساحة بالكود.

</div>

In [ ]:
# اكتب حلّك هنا
